# GVHMR on Google Colab — fixed & working (October 2026)

**GVHMR** ([paper](https://arxiv.org/abs/2409.06662) · [code](https://github.com/zju3dv/GVHMR), SIGGRAPH Asia 2024) turns a normal video of a person into 3D human motion:
the body mesh **in the camera view** (left) and the same motion **in a ground-aligned 3D world** (right).

The official Colab notebook stopped working because Colab now runs **Python 3.13** (Ubuntu 24.04), and GVHMR depends on packages that only exist for Python 3.10.
This notebook fixes that. The **"What was fixed"** section at the bottom explains each change.

---

## ▶️ How to use it (no coding needed)

| | What to do |
|---|---|
| **1** | Menu **Runtime ▸ Change runtime type** ▸ choose **T4 GPU** ▸ **Save** |
| **2** | Menu **Runtime ▸ Run all**. Setup takes about **8–10 minutes** the first time in each session |
| **3** | At **Step 5**, click **Choose files** and pick your own video (`.mp4`, `.mov`, …). The result plays under the cell |
| **4** | Run **Step 6** to download your results (videos + 3D motion data) |

💡 **The free Colab tier is enough.** It's just slower, and GPU time per day is limited. Colab Pro gives faster GPUs (L4 / A100) and longer sessions, but you don't need it.

💡 Every time Colab gives you a **new session** (e.g. the next day), run Steps 1–3 again. Colab erases the machine when a session ends.

## Step 1 — Check the GPU

In [ ]:
#@title 1️⃣ Check that a GPU is connected { display-mode: "form" }
import subprocess
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True, check=True).stdout.strip()
    print("✅ GPU found:", gpu)
except Exception:
    raise SystemExit("❌ No GPU found.\n"
                     "   Fix: menu Runtime ▸ Change runtime type ▸ Hardware accelerator = T4 GPU ▸ Save,\n"
                     "   then run this notebook again (Runtime ▸ Run all).")

## Step 2 — Install GVHMR (≈ 4–6 min)

Colab's own Python is 3.13. GVHMR needs **Python 3.10 + PyTorch 2.3 (CUDA 12.1) + numpy 1.23.5 + pytorch3d 0.7.6**.
This cell creates a **separate Python 3.10 environment** in `/content/gvhmr-env` using [`uv`](https://github.com/astral-sh/uv) and installs the exact package versions GVHMR was tested with.
It doesn't touch Colab's own packages, so Colab won't ask you to restart.

In [ ]:
#@title 2️⃣ Install GVHMR + Python 3.10 environment { display-mode: "form" }
import os, sys, time, codecs, select, subprocess
from pathlib import Path

GVHMR_COMMIT = "ee960bb6e2ea2d381aa97f08e9b71ef320b624b1"  # pinned so future changes on GitHub can't break this notebook
REPO, ENV = "/content/GVHMR", "/content/gvhmr-env"
LOCK_FILE = "/content/gvhmr_requirements_colab.txt"

# Exact, tested versions (resolved for Linux + Python 3.10). chumpy is installed separately below.
Path(LOCK_FILE).write_text("""
--index-url https://pypi.org/simple
--extra-index-url https://download.pytorch.org/whl/cu121
aiohappyeyeballs==2.6.1
aiohttp==3.12.6
aiosignal==1.3.2
antlr4-python3-runtime==4.9.3
async-timeout==5.0.1
attrs==25.3.0
autocommand==2.2.2
av==13.0.0
backports-tarfile==1.2.0
certifi==2025.4.26
charset-normalizer==3.4.2
cheroot==10.0.1
cherrypy==18.10.0
colorlog==6.9.0
contourpy==1.3.2
cycler==0.12.1
cython==3.1.1
cython-bbox==0.1.5
einops==0.8.1
ffmpeg-python==0.2.0
filelock==3.18.0
fonttools==4.58.1
frozenlist==1.6.0
fsspec==2025.5.1
future==1.0.0
fvcore==0.1.5.post20221221
hf-xet==1.1.2
huggingface-hub==0.32.3
hydra-colorlog==1.2.0
hydra-core==1.3.0
hydra-zen==0.14.0
idna==3.10
imageio==2.34.1
iopath==0.1.10
jaraco-collections==5.1.0
jaraco-context==6.0.1
jaraco-functools==4.1.0
jaraco-text==4.0.0
jinja2==3.1.6
joblib==1.5.1
kiwisolver==1.4.8
lapx==0.5.11.post1
lazy-loader==0.4
lightning==2.3.0
lightning-utilities==0.14.3
markdown-it-py==3.0.0
markupsafe==3.0.2
matplotlib==3.10.3
mdurl==0.1.2
more-itertools==10.7.0
mpmath==1.3.0
multidict==6.4.4
networkx==3.4.2
numpy==1.23.5
nvidia-cublas-cu12==12.1.3.1
nvidia-cuda-cupti-cu12==12.1.105
nvidia-cuda-nvrtc-cu12==12.1.105
nvidia-cuda-runtime-cu12==12.1.105
nvidia-cudnn-cu12==8.9.2.26
nvidia-cufft-cu12==11.0.2.54
nvidia-curand-cu12==10.3.2.106
nvidia-cusolver-cu12==11.4.5.107
nvidia-cusparse-cu12==12.1.0.106
nvidia-nccl-cu12==2.20.5
nvidia-nvjitlink-cu12==12.9.41
nvidia-nvtx-cu12==12.1.105
omegaconf==2.3.0
opencv-python==4.11.0.86
packaging==24.2
pandas==2.2.3
pillow==11.2.1
portalocker==3.1.1
portend==3.2.1
propcache==0.3.1
protobuf==6.31.1
psutil==7.0.0
py-cpuinfo==9.0.0
pycolmap==3.11.1
pygments==2.19.1
pyparsing==3.2.3
python-dateutil==2.9.0.post0
pytorch-lightning==2.3.0
pytorch3d @ https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt230/pytorch3d-0.7.6-cp310-cp310-linux_x86_64.whl
pytz==2025.2
pyyaml==6.0.2
requests==2.32.3
rich==14.0.0
safetensors==0.5.3
scikit-image==0.24.0
scipy==1.15.3
seaborn==0.13.2
setuptools==80.9.0
six==1.17.0
smplx==0.1.28
sympy==1.14.0
tabulate==0.9.0
tempora==5.8.0
tensorboardx==2.6.2.2
termcolor==3.1.0
tifffile==2025.5.10
timm==0.9.12
torch==2.3.0+cu121
torchmetrics==1.7.2
torchvision==0.18.0+cu121
tqdm==4.67.1
transforms3d==0.4.2
trimesh==4.6.10
triton==2.3.0
typing-extensions==4.13.2
tzdata==2025.2
ultralytics==8.2.42
ultralytics-thop==2.0.14
urllib3==2.4.0
wis3d==1.0.1
yacs==0.1.8
yarl==1.20.0
zc-lockfile==3.0.post1
""")

def _clean_env():
    env = dict(os.environ)
    env.pop("PYTHONPATH", None)  # Colab's PYTHONPATH (/env/python) points at its own Python 3.13 packages
    # torch 2.3 ships its own CUDA 12.1 libraries: hide Colab's newer CUDA toolkit libs, keep the GPU driver
    env["LD_LIBRARY_PATH"] = ":".join(p for p in env.get("LD_LIBRARY_PATH", "").split(":") if p and "cuda" not in p.lower())
    env["MPLBACKEND"] = "Agg"  # Colab's inline plot backend doesn't exist in the 3.10 env
    env["PYTHONUNBUFFERED"] = "1"
    env["UV_LINK_MODE"] = "copy"
    env["UV_PYTHON_PREFERENCE"] = "only-managed"
    return env

def sh(cmd, cwd=None):
    """Run a bash command, show its output live, stop the notebook if it fails."""
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", cwd=cwd, env=_clean_env(),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    dec, t = codecs.getincrementaldecoder("utf-8")(errors="replace"), time.time()
    while True:
        if not select.select([p.stdout], [], [], 20)[0]:  # quiet for 20 s -> show we're still alive
            print(f"      ... still working ({time.time() - t:.0f} s)", flush=True)
            continue
        chunk = p.stdout.read1(65536)
        if not chunk:
            break
        sys.stdout.write(dec.decode(chunk)); sys.stdout.flush()
    if p.wait() != 0:
        raise SystemExit(f"\n❌ Installation step failed (exit code {p.returncode}). See the messages above.")

t0 = time.time()
print("📦 [1/5] Installing tools (uv, aria2) ...")
sh("command -v uv >/dev/null || pip install -q uv")
sh("command -v aria2c >/dev/null || (apt-get -qq install -y aria2 || (apt-get -qq update && apt-get -qq install -y aria2)) >/dev/null 2>&1 || true")

print("📥 [2/5] Downloading GVHMR source code ...")
sh(f"""set -e
[ -d {REPO}/.git ] || git clone -q https://github.com/zju3dv/GVHMR {REPO}
cd {REPO} && git checkout -q {GVHMR_COMMIT} && git log -1 --format='      GVHMR commit %h (%cd)' --date=short""")

print("🐍 [3/5] Creating the Python 3.10 environment ...")
sh(f"[ -x {ENV}/bin/python ] || uv venv -q --seed --python 3.10 {ENV}; {ENV}/bin/python --version")

print("📚 [4/5] Installing packages (torch 2.3 + CUDA 12.1, pytorch3d, ...). This is the slow part ...")
sh(f"uv pip install -q --python {ENV}/bin/python --index-strategy unsafe-best-match -r {LOCK_FILE}")
# chumpy (needed to read the SMPL body model) has a broken setup.py that imports pip -> build it without isolation
sh(f"uv pip install -q --python {ENV}/bin/python --no-build-isolation chumpy==0.70")
sh(f"uv pip install -q --python {ENV}/bin/python --no-deps -e {REPO}")

print("🔎 [5/5] Checking the installation ...")
sh(f"""{ENV}/bin/python -W ignore - <<'EOF'
import torch, pytorch3d, numpy, chumpy, smplx, ultralytics, pycolmap, hmr4d
from pytorch3d.renderer import MeshRasterizer
print(f"      python {{__import__('sys').version.split()[0]}} | torch {{torch.__version__}} | pytorch3d {{pytorch3d.__version__}} | numpy {{numpy.__version__}}")
ok = torch.cuda.is_available()
print(f"      CUDA available: {{ok}}" + (f" ({{torch.cuda.get_device_name(0)}})" if ok else "  <-- switch the runtime to T4 GPU!"))
EOF""")
print(f"\n✅ Installation finished in {(time.time()-t0)/60:.1f} min")

## Step 3 — Download the pretrained models (≈ 5.8 GB, 1–3 min)

These are the same files the official GVHMR Colab uses: the GVHMR network, YOLOv8 (person detector), ViTPose (2D keypoints), HMR2 (image features) and the SMPL / SMPL-X body models.

In [ ]:
#@title 3️⃣ Download model weights { display-mode: "form" }
import os, subprocess
from pathlib import Path

CKPT = Path("/content/GVHMR/inputs/checkpoints")
HF = "https://huggingface.co"
WEIGHTS = [  # (url, destination, expected size in bytes)
    (f"{HF}/camenduru/GVHMR/resolve/main/gvhmr/gvhmr_siga24_release.ckpt", "gvhmr/gvhmr_siga24_release.ckpt", 163508011),
    (f"{HF}/camenduru/GVHMR/resolve/main/hmr2/epoch%3D10-step%3D25000.ckpt", "hmr2/epoch=10-step=25000.ckpt", 2709494041),
    (f"{HF}/camenduru/GVHMR/resolve/main/vitpose/vitpose-h-multi-coco.pth", "vitpose/vitpose-h-multi-coco.pth", 2549075546),
    (f"{HF}/camenduru/GVHMR/resolve/main/yolo/yolov8x.pt", "yolo/yolov8x.pt", 136867539),
    (f"{HF}/camenduru/SMPLer-X/resolve/main/SMPL_NEUTRAL.pkl", "body_models/smpl/SMPL_NEUTRAL.pkl", 247186228),
    (f"{HF}/camenduru/SMPLer-X/resolve/main/SMPLX_NEUTRAL.npz", "body_models/smplx/SMPLX_NEUTRAL.npz", 108752058),
]

if not Path("/content/GVHMR").exists():
    raise SystemExit("❌ Run Step 2 first.")
import shutil, time
use_aria2 = shutil.which("aria2c") is not None
for url, rel, size in WEIGHTS:
    dst = CKPT / rel
    if dst.exists() and dst.stat().st_size == size:
        print(f"✔ already there   {rel}")
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    print(f"⬇ downloading     {rel}  ({size/1e9:.2f} GB) ... ", end="", flush=True)
    t = time.time()
    for attempt in range(3):
        if use_aria2:  # 16 parallel connections = fast
            cmd = ["aria2c", "--console-log-level=error", "--summary-interval=0", "--download-result=hide", "-q",
                   "-c", "-x", "16", "-s", "16", "-k", "1M", url, "-d", str(dst.parent), "-o", dst.name]
        else:
            cmd = ["curl", "-sSL", "--retry", "5", "-C", "-", "-o", str(dst), url]
        subprocess.run(cmd)
        if dst.exists() and dst.stat().st_size == size:
            print(f"done in {time.time()-t:.0f} s")
            break
        print(f"\n   retrying ({attempt+1}/3) ... ", end="", flush=True)
    else:
        raise SystemExit(f"\n❌ Could not download {rel}. Please run this cell again.")
print("\n✅ All model weights are ready.")

## Step 4 — Load the GVHMR runner and test it on the example video (≈ 2 min)

This defines `run_gvhmr(...)` and runs it on GVHMR's own tennis example. If you see the tennis result video below, everything works.

In [ ]:
#@title 4️⃣ Load the runner + run the example (tennis) { display-mode: "form" }
import os, re, sys, time, json, shutil, base64, codecs, subprocess
from pathlib import Path
from IPython.display import HTML, display

REPO = "/content/GVHMR"
ENV_PY = "/content/gvhmr-env/bin/python"
RESULTS = Path("/content/gvhmr_results")
LAST_RESULT = None

def _clean_env():
    env = dict(os.environ)
    env.pop("PYTHONPATH", None)
    env["LD_LIBRARY_PATH"] = ":".join(p for p in env.get("LD_LIBRARY_PATH", "").split(":") if p and "cuda" not in p.lower())
    env["MPLBACKEND"] = "Agg"
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONWARNINGS"] = "ignore"  # hide harmless deprecation warnings (e.g. pkg_resources)
    return env

def _stream(cmd, cwd=None, log_path=None):
    """Run a command, print its output live (progress bars included), return (exit_code, output_tail)."""
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", cwd=cwd, env=_clean_env(),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    dec, tail = codecs.getincrementaldecoder("utf-8")(errors="replace"), ""
    log = open(log_path, "w") if log_path else None
    while chunk := p.stdout.read1(65536):
        text = dec.decode(chunk)
        sys.stdout.write(text); sys.stdout.flush()
        if log: log.write(text)
        tail = (tail + text)[-20000:]
    if log: log.close()
    return p.wait(), tail

def _probe(path):
    out = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-count_packets",
                          "-show_entries", "stream=width,height,nb_read_packets", "-of", "json", str(path)],
                         capture_output=True, text=True)
    s = json.loads(out.stdout or "{}").get("streams", [])
    if not s:
        raise SystemExit(f"❌ Could not read a video stream from {path}. Is it a valid video file?")
    return int(s[0]["width"]), int(s[0]["height"]), int(s[0]["nb_read_packets"])

def _safe_name(stem):
    name = re.sub(r"[^A-Za-z0-9_-]+", "_", stem).strip("_-")[:60] or "my_video"
    return name if name[0].isalpha() else "v_" + name   # Hydra would turn names like 123 / 1e5 into numbers

def prepare_video(src, name, max_side=1920, start=0, duration=0):
    """GVHMR assumes 30 FPS input. Convert to 30 FPS, apply phone rotation, cap the resolution, drop audio."""
    dst = Path(REPO) / "inputs" / "demo" / f"{name}.mp4"
    dst.parent.mkdir(parents=True, exist_ok=True)
    if max_side:
        m = int(max_side)
        vf = (f"fps=30,scale=w='if(gte(iw,ih),trunc(min({m},iw)/2)*2,-2)'"
              f":h='if(gte(iw,ih),-2,trunc(min({m},ih)/2)*2)'")
    else:
        vf = "fps=30,scale=trunc(iw/2)*2:trunc(ih/2)*2"
    cmd = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error"]
    if start:
        cmd += ["-ss", str(start)]
    cmd += ["-i", str(src)]
    if duration:
        cmd += ["-t", str(duration)]
    cmd += ["-vf", vf, "-an", "-c:v", "libx264", "-preset", "veryfast", "-crf", "17", "-pix_fmt", "yuv420p", str(dst)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0 or not dst.exists():
        raise SystemExit(f"❌ ffmpeg could not convert the video:\n{r.stderr[-2000:]}")
    return dst

def show_video(path, max_width=1280):
    """Embed a small preview of a (possibly huge) mp4 in the notebook."""
    path = Path(path)
    preview = path.with_name(path.stem + "_preview.mp4")
    subprocess.run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(path), "-vf",
                    f"scale='min({max_width},iw)':-2", "-c:v", "libx264", "-preset", "veryfast", "-crf", "28",
                    "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(preview)], check=True)
    size_mb = preview.stat().st_size / 1e6
    if size_mb > 40:
        print(f"(Preview is {size_mb:.0f} MB, too big to show here. Download the results in Step 6.)")
        return
    data = base64.b64encode(preview.read_bytes()).decode()
    preview.unlink()
    display(HTML(f'<video controls loop autoplay muted playsinline style="max-width:100%;width:{max_width}px" '
                 f'src="data:video/mp4;base64,{data}"></video>'))

EXPORT_NPZ = r"""
import sys, torch, numpy as np
pred = torch.load(sys.argv[1], map_location="cpu")
out = {"fps": np.array(30), "K_fullimg": pred["K_fullimg"].numpy()}
for space in ("global", "incam"):
    for k, v in pred[f"smpl_params_{space}"].items():
        out[f"{space}_{k}"] = v.numpy()
np.savez(sys.argv[2], **out)
print("      " + ", ".join(f"{k}{tuple(v.shape)}" for k, v in out.items()))
"""

def _explain_failure(code, log):
    print("\n" + "=" * 70)
    if code in (-9, 137) or "Killed" in log[-3000:]:
        print("❌ Colab ran out of RAM. Use a shorter clip (duration_seconds) or a lower max_resolution.")
    elif "CUDA out of memory" in log or "OutOfMemoryError" in log:
        print("❌ The GPU ran out of memory. Use a shorter clip or a lower max_resolution.")
    elif "id_sorted[0]" in log or "list index out of range" in log:
        print("❌ No person was detected in the video. GVHMR needs one clearly visible person.")
    elif "no CUDA" in log or "NVIDIA driver" in log or "Torch not compiled with CUDA" in log:
        print("❌ No GPU. Runtime ▸ Change runtime type ▸ T4 GPU, then run Steps 1–3 again.")
    else:
        print("❌ GVHMR stopped with an error. The last lines above show the reason.")
    print("=" * 70)

def run_gvhmr(video_path, static_camera=False, focal_length_mm=0, max_side=1920, start=0, duration=0, name=None):
    """Run the full GVHMR demo on one video and collect the results in /content/gvhmr_results/<name>/."""
    global LAST_RESULT
    if not Path(ENV_PY).exists():
        raise SystemExit("❌ Run Step 2 (install) and Step 3 (download weights) first.")
    src = Path(video_path)
    if not src.exists():
        raise SystemExit(f"❌ Video not found: {src}")
    name = name or _safe_name(src.stem)
    t0 = time.time()

    print(f"[1/4] 🎞️  Preparing '{src.name}' (30 FPS, max {max_side or 'original'} px) ...")
    inp = prepare_video(src, name, max_side, start, duration)
    w, h, n = _probe(inp)
    print(f"      {w}x{h}, {n} frames = {n/30:.1f} s")
    if n / 30 * (w * h) / (1920 * 1080) > 120:
        print("      ⚠️ Long / high-resolution video: free Colab may run out of RAM. "
              "If it fails, use duration_seconds or max_resolution=1280.")

    out_dir = Path(REPO) / "outputs" / "demo" / name
    shutil.rmtree(out_dir, ignore_errors=True)  # start fresh, otherwise GVHMR reuses old cached results
    print(f"[2/4] 💃 Running GVHMR ({'static camera' if static_camera else 'moving camera → visual odometry'}) ...")
    cmd = f"{ENV_PY} tools/demo/demo.py --video='{inp}'"
    cmd += " -s" if static_camera else ""
    cmd += f" --f_mm={int(focal_length_mm)}" if focal_length_mm else ""
    RESULTS.mkdir(parents=True, exist_ok=True)
    code, log = _stream(cmd, cwd=REPO, log_path=RESULTS / f"{name}_log.txt")
    side_by_side = out_dir / f"{name}_3_incam_global_horiz.mp4"
    if code != 0 or not side_by_side.exists():
        _explain_failure(code, log)
        raise SystemExit(f"GVHMR failed (full log: {RESULTS / (name + '_log.txt')})")

    print("[3/4] 📦 Collecting results ...")
    res_dir = RESULTS / name
    shutil.rmtree(res_dir, ignore_errors=True)
    res_dir.mkdir(parents=True)
    files = {
        "side_by_side": (side_by_side, f"{name}_side_by_side.mp4"),
        "incam": (out_dir / "1_incam.mp4", f"{name}_incam.mp4"),
        "global": (out_dir / "2_global.mp4", f"{name}_global.mp4"),
        "results_pt": (out_dir / "hmr4d_results.pt", f"{name}_hmr4d_results.pt"),
    }
    result = {"name": name, "dir": str(res_dir)}
    for key, (srcf, dstname) in files.items():
        shutil.copy(srcf, res_dir / dstname)
        result[key] = str(res_dir / dstname)
    npz = res_dir / f"{name}_smplx_params.npz"
    Path("/content/gvhmr_export_npz.py").write_text(EXPORT_NPZ)
    _stream(f"{ENV_PY} -W ignore /content/gvhmr_export_npz.py '{result['results_pt']}' '{npz}'")
    result["npz"] = str(npz)
    shutil.move(str(RESULTS / f"{name}_log.txt"), res_dir / f"{name}_log.txt")
    result["zip"] = shutil.make_archive(str(RESULTS / f"GVHMR_{name}"), "zip", res_dir)
    LAST_RESULT = result
    print(f"      saved in {res_dir}   (zip: {result['zip']})")

    print(f"[4/4] 📺 Done in {(time.time()-t0)/60:.1f} min. Left: camera view · Right: 3D world view")
    show_video(side_by_side)
    return result

# ---- Example: GVHMR's own tennis clip (the camera is static, so static_camera=True) ----
example = run_gvhmr(f"{REPO}/docs/example_video/tennis.mp4", static_camera=True)

## Step 5 — Run GVHMR on **your own video** 🎥

1. Choose the options in the form (the defaults are fine for most videos).
2. Press ▶ on the cell. With **"Upload from my computer"**, a **Choose files** button appears under the cell. Click it and pick your video.
3. Wait. Progress bars show each stage. On a free T4, a 10-second clip takes about 3–4 minutes, and a 60-second clip about 10–15 minutes.

**Big video (over ~100 MB)?** Use the Files panel instead, it's faster and more reliable: click the 📁 **folder icon** on the left, drag your video into that panel, wait until the upload circle finishes, then choose **"Files panel (drag & drop)"** below and press ▶.

⚠️ Colab runs each cell with the settings it had **when you pressed Run all**. If you change this form afterwards, just press ▶ on Step 5 again.

**Tips for good results:** one person, whole body visible, not too small in the frame. Any frame rate or resolution works (the notebook converts it).

In [ ]:
#@title 5️⃣ Run GVHMR on YOUR video { display-mode: "form" }
#@markdown ### Where is your video?
video_source = "Upload from my computer"  #@param ["Upload from my computer", "Files panel (drag & drop)", "Google Drive (file path)", "Google Drive (share link)"]
#@markdown Only for the other options: **Files panel** → file name (empty = the newest video you uploaded) · **Drive path** → e.g. `MyDrive/videos/dance.mp4` · **Share link** → `https://drive.google.com/file/d/XXXX/view?usp=sharing`
file_name_path_or_link = ""  #@param {type:"string"}
#@markdown ---
#@markdown ### Was the camera completely still (tripod)?
#@markdown ✅ tick it for a tripod / fixed camera (faster and more accurate). ⬜ Leave it unticked for a hand-held or moving camera.
camera_is_static = False  #@param {type:"boolean"}
#@markdown ---
#@markdown ### Optional settings (the defaults are fine)
max_resolution = "1920 (Full HD, recommended)"  #@param ["1280 (faster, less memory)", "1920 (Full HD, recommended)", "Original (may run out of memory)"]
#@markdown Process only part of the video: start time and length in seconds (0 = the whole video)
start_at_second = 0  #@param {type:"number"}
duration_seconds = 0  #@param {type:"number"}
#@markdown Focal length of the camera in mm, full-frame equivalent (0 = automatic). iPhone lenses 0.5x/1x/2x/3x ≈ 13/24/48/77 mm
focal_length_mm = 0  #@param {type:"integer"}

import os, shutil, subprocess
from pathlib import Path

if "run_gvhmr" not in globals():
    raise SystemExit("❌ Please run Steps 2, 3 and 4 first (Runtime ▸ Run all).")

UPLOADS = Path("/content/uploads")
UPLOADS.mkdir(exist_ok=True)
src = None

VIDEO_EXT = {".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm", ".mpg", ".mpeg", ".wmv", ".3gp"}

if video_source.startswith("Upload"):
    from google.colab import files
    print("👉 Click 'Choose files' below and select your video (one file).")
    uploaded = files.upload(target_dir=str(UPLOADS))
    if not uploaded:
        raise SystemExit("❌ No file was uploaded.")
    src = Path(list(uploaded.keys())[-1])
    src = src if src.is_absolute() else UPLOADS / src

elif video_source.startswith("Files panel"):
    wanted = file_name_path_or_link.strip()
    if wanted:
        src = Path(wanted if wanted.startswith("/") else "/content/" + wanted)
    else:
        videos = [f for d in (Path("/content"), UPLOADS) for f in d.iterdir()
                  if f.is_file() and f.suffix.lower() in VIDEO_EXT]
        if not videos:
            raise SystemExit("❌ No video found in the Files panel. Click the 📁 icon on the left, drag your "
                             "video into the panel and wait until the upload finishes.")
        src = max(videos, key=lambda f: f.stat().st_mtime)

elif video_source == "Google Drive (file path)":
    from google.colab import drive
    drive.mount("/content/drive")
    p = file_name_path_or_link.strip().strip('"').strip("'")
    if not p:
        raise SystemExit("❌ Paste the path of your video in 'file_name_path_or_link' (e.g. MyDrive/videos/dance.mp4).")
    src = Path(p if p.startswith("/") else "/content/drive/" + p.removeprefix("drive/"))

else:  # share link
    link = file_name_path_or_link.strip()
    if "drive.google.com" not in link:
        raise SystemExit("❌ Paste a Google Drive share link in 'file_name_path_or_link'. "
                         "(In Drive: right-click the video ▸ Share ▸ 'Anyone with the link' ▸ Copy link.)")
    try:
        import gdown
    except ImportError:
        subprocess.run("pip install -q gdown", shell=True, check=True); import gdown
    out = gdown.download(link, str(UPLOADS) + "/", fuzzy=True, quiet=False)
    if not out:
        raise SystemExit("❌ Could not download from that link. In Google Drive set sharing to 'Anyone with the link'.")
    src = Path(out)

if not src.exists():
    raise SystemExit(f"❌ File not found: {src}")
print(f"🎬 Input video: {src}  ({src.stat().st_size/1e6:.1f} MB)\n")
max_side = {"1280": 1280, "1920": 1920}.get(max_resolution.split()[0], 0)
result = run_gvhmr(src, static_camera=camera_is_static, focal_length_mm=focal_length_mm,
                   max_side=max_side, start=start_at_second, duration=duration_seconds)

## Step 6 — Download your results 💾

You get one `.zip` file with:

| File | What it is |
|---|---|
| `<name>_side_by_side.mp4` | camera view + 3D world view next to each other |
| `<name>_incam.mp4` | the 3D body drawn on top of your video |
| `<name>_global.mp4` | the motion in a 3D world with a ground plane |
| `<name>_smplx_params.npz` | the 3D motion as numbers (SMPL-X body pose, shape, rotation, translation per frame, 30 FPS) for Blender / Unity / your own code |
| `<name>_hmr4d_results.pt` | GVHMR's raw output (PyTorch) |
| `<name>_log.txt` | the full processing log |

In [ ]:
#@title 6️⃣ Download the results of the last run { display-mode: "form" }
also_save_to_google_drive = False  #@param {type:"boolean"}
import os
from google.colab import files
if not globals().get("LAST_RESULT"):
    raise SystemExit("❌ Nothing to download yet. Run Step 5 (or Step 4) first.")
zip_path = LAST_RESULT["zip"]
if also_save_to_google_drive:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/GVHMR_results", exist_ok=True)
    shutil.copy(zip_path, "/content/drive/MyDrive/GVHMR_results/")
    print("✅ Saved to Google Drive ▸ My Drive ▸ GVHMR_results")
print(f"⬇ Downloading {os.path.basename(zip_path)} ({os.path.getsize(zip_path)/1e6:.1f} MB) ...")
files.download(zip_path)

---
## 🛠️ What was broken and how it was fixed

On today's Colab the official notebook fails at the very first install step:

```
ERROR: pytorch3d-0.7.6-cp310-cp310-linux_x86_64.whl is not a supported wheel on this platform.
ERROR: No matching distribution found for torch==2.3.0+cu121
ModuleNotFoundError: No module named 'pytorch_lightning'
```

**1. Colab now runs Python 3.13** (Ubuntu 24.04); the official notebook was made in 2024 for Python 3.10. GVHMR pins `torch 2.3.0`, `numpy 1.23.5` and a `pytorch3d` wheel that only exist for Python 3.10, so `pip install -r requirements.txt` installs nothing and `demo.py` can't start.
→ **Fix:** a separate **Python 3.10 environment** (`/content/gvhmr-env`, created with `uv`) runs GVHMR. Colab's own Python is left alone, so there are no "restart session" prompts.

**2. `chumpy` can't be built** by modern pip/uv: its `setup.py` imports `pip` and uses `distutils`. chumpy is needed to read `SMPL_NEUTRAL.pkl`.
→ **Fix:** install `chumpy==0.70` in the 3.10 env with `--no-build-isolation`.

**3. Unpinned dependencies drifted.** `setuptools>=68.0` now resolves to setuptools 84, which no longer contains `pkg_resources`. Lightning 2.3 runs `__import__("pkg_resources")` on import, so `import pytorch_lightning` crashes even on Python 3.10. Many other packages are unpinned too (pycolmap, opencv, huggingface-hub, …).
→ **Fix:** all 119 packages are pinned to versions that resolve together (e.g. `setuptools==80.9.0`, `pycolmap==3.11.1`, `pytorch-lightning==2.3.0`). GVHMR is pinned to commit `ee960bb`. Dev-only tools (jupyter, ipdb, black) are skipped.

**4. Colab's environment variables leak into the 3.10 env.** `PYTHONPATH=/env/python` points at Colab's 3.13 packages, and `MPLBACKEND` at a plotting backend that doesn't exist in the 3.10 env.
→ **Fix:** GVHMR runs with a clean environment (no `PYTHONPATH`, no CUDA-toolkit library paths, `MPLBACKEND=Agg`).

**5. Custom videos.** The official notebook only accepts a Google Drive share link. File names with spaces or brackets break GVHMR's Hydra config (`my video (1).mp4`). 60 FPS phone videos are processed as if they were 30 FPS (wrong speed), and 4K videos run out of RAM on free Colab.
→ **Fix:** upload button, Files-panel drag & drop, Drive path or share link. File names are cleaned. Videos are converted to **30 FPS**, phone rotation is applied, resolution is capped at 1920 px, and you can process just part of a video.

**6. Re-running the same video silently reused old cached results.** → **Fix:** the output folder is cleared before each run.

**7. No easy way to get the results out.** → **Fix:** a light preview in the notebook, plus one zip with the videos, `.npz` motion data and raw `.pt` (one-click download or Google Drive copy), and clear messages for "no person found", "out of memory" and "no GPU".

## ❓ Troubleshooting

* **"No GPU found"**: Runtime ▸ Change runtime type ▸ **T4 GPU** ▸ Save, then Runtime ▸ Run all.
* **"Cannot connect to GPU backend" / usage limits**: the free GPU quota for today is used up. Try again later, or use Colab Pro.
* **Session disconnected / everything is gone**: Colab erases the machine after a session ends (≈ 90 min idle or 12 h max). Run all steps again. Save your results to Google Drive (Step 6) so you keep them.
* **Out of memory**: use a shorter clip (`duration_seconds`) or `max_resolution = 1280`.
* **Wrong person tracked**: GVHMR follows the person who is biggest / visible the longest in the video. Crop or trim your video so the main person dominates.
* **Hand-held phone video**: leave **camera_is_static** unticked. GVHMR then estimates the camera rotation (SimpleVO).

## 📄 Licenses

GVHMR is free for **non-commercial research and education** ([license](https://github.com/zju3dv/GVHMR/blob/main/LICENSE)); for commercial use contact the authors.
The SMPL / SMPL-X body models have their own non-commercial licenses ([SMPL](https://smpl.is.tue.mpg.de/), [SMPL-X](https://smpl-x.is.tue.mpg.de/)). Please register there if you use them.
The model files are downloaded from the same Hugging Face mirror the official GVHMR Colab uses.